In [ ]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

# Define um estilo visual mais agradável para os gráficos
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported and style set.")

In [ ]:
# Célula 2 (Com as chaves de leitura corrigidas)

import pandas as pd
import os
import glob

results_dir = 'results'
experiment_data = []

# A busca pelos arquivos já está correta
stat_files = glob.glob(os.path.join(results_dir, '**', '*_stats.txt'), recursive=True)

for file_path in stat_files:
    data = {}
    with open(file_path, 'r') as f:
        next(f, None) # Pula a primeira linha
        for line in f:
            if ':' in line:
                key, value = line.split(':', 1)
                # Remove espaços em branco da chave para garantir consistência
                data[key.strip()] = value.strip()
    
    # Extrai o nome base da instância a partir do diretório pai do arquivo de stats
    instance_info = os.path.basename(os.path.dirname(file_path))
    data['instance_base'] = instance_info.split('_lambda_')[0]
    
    experiment_data.append(data)

df_results = pd.DataFrame(experiment_data)

# --- CORREÇÃO APLICADA AQUI ---
# Converte as colunas para o tipo numérico usando as NOVAS chaves
df_results['lambda'] = pd.to_numeric(df_results['Lambda_Weight'])
df_results['combined_obj_z'] = pd.to_numeric(df_results['Combined_Objective_Value_Z'])
df_results['disagreement_f1'] = pd.to_numeric(df_results['Disagreement_Value_f1'])
df_results['num_clusters_f2'] = pd.to_numeric(df_results['Num_Clusters_Value_f2'])
df_results['execution_time_s'] = pd.to_numeric(df_results['Solver_Execution_Time_s'])

# Ordenação natural para P1, P2, ..., P10
df_results['instance_num'] = df_results['instance_base'].str.extract(r'P(\d+)').astype(int)
df_results.sort_values(by=['instance_num', 'lambda'], inplace=True)
df_results.drop(columns=['instance_num'], inplace=True)

print(f"Loaded and processed {len(df_results)} experiment results successfully.")

# Exibe as colunas relevantes para verificação
display(df_results[['instance_base', 'lambda', 'disagreement_f1', 'num_clusters_f2', 'execution_time_s']])

In [ ]:
# Célula 3 (Com salvamento de gráficos em uma pasta dedicada)

# Pega uma lista ordenada de todas as instâncias únicas
unique_instances = sorted(df_results['instance_base'].unique())

# --- ALTERAÇÃO: Definir e criar a pasta de saída para os gráficos ---
plots_dir = os.path.join('results', 'plots')
os.makedirs(plots_dir, exist_ok=True)
print(f"Os gráficos serão salvos no diretório: '{plots_dir}'")
# --------------------------------------------------------------------

# Itera sobre cada nome de instância na lista
for instance_name in unique_instances:
    df_instance = df_results[df_results['instance_base'] == instance_name].copy()

    print("="*80)
    print(f"ANALYSIS FOR INSTANCE: {instance_name}")
    print("="*80)

    # --- 1. Plot da Fronteira de Pareto CORRETA ---
    fig, ax = plt.subplots(figsize=(12, 8))
    
    ax.plot(
        df_instance['num_clusters_f2'], # Eixo X: Número de Clusters (Simplicidade)
        df_instance['disagreement_f1'],  # Eixo Y: Desequilíbrio (Qualidade)
        marker='o', 
        markersize=10,
        linestyle='--',
        color='b',
        label='Soluções de Trade-off'
    )

    for i, row in df_instance.iterrows():
        ax.text(
            row['num_clusters_f2'] + 0.02, # Posição da anotação no eixo X
            row['disagreement_f1'],      # Posição da anotação no eixo Y
            f" λ={row['lambda']}", 
            fontsize=12,
            verticalalignment='bottom'
        )

    ax.set_xlabel('Número de Clusters (k) - (Simplicidade)', fontsize=14)
    ax.set_ylabel('Desequilíbrio Esperado (f1) - (Qualidade)', fontsize=14)
    ax.set_title(f'Fronteira de Pareto para a Instância: {instance_name}', fontsize=16, pad=20)
    ax.legend(fontsize=12)
    ax.grid(True)
    
    unique_k_values = sorted(df_instance['num_clusters_f2'].unique())
    if len(unique_k_values) > 1:
        ax.set_xticks(unique_k_values)

    # --- ALTERAÇÃO: Usar o novo diretório de plots ---
    output_fig_path = os.path.join(plots_dir, f'pareto_{instance_name}.png')
    plt.savefig(output_fig_path, dpi=300)
    print(f"Pareto front plot saved to '{output_fig_path}'")
    plt.show()

    # --- 2. Análise das Soluções de Destaque ---
    print(f"\n--- Soluções de Destaque para: {instance_name} ---")
    display(df_instance[['lambda', 'disagreement_f1', 'num_clusters_f2']].rename(columns={'disagreement_f1': 'Desequilíbrio', 'num_clusters_f2': 'Num_Clusters'}))
    print("\n\n")